# Program 2: Visualising word embeddings with PCA and t-SNE

**Aim.** Reduce embeddings of 10 domain words to two dimensions, plot them, and analyse clusters and relationships.

**Theory.** A 100-dimensional vector cannot be drawn. PCA projects onto the directions of greatest variance and keeps global distances. t-SNE keeps local neighbourhoods and is better for showing clusters, but its axes have no meaning and it depends on the random seed and the perplexity.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
import numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-100')
medical = ['doctor', 'nurse', 'hospital', 'medicine', 'patient', 'tablet', 'surgery', 'fever', 'pharmacy', 'vaccine']
sports = ['cricket', 'football', 'tennis', 'stadium', 'goal', 'coach', 'match', 'player', 'tournament', 'referee']
print(len(medical), 'medical words;', len(sports), 'sports words for contrast')

10 medical words; 10 sports words for contrast


In [3]:
words = medical + sports
X = np.array([wv[w] for w in words])
pca = PCA(n_components=2, random_state=0).fit(X)
P = pca.transform(X)
T = TSNE(n_components=2, perplexity=5, random_state=0, init='pca').fit_transform(X)
print('variance kept by the 2 PCA axes:', round(float(pca.explained_variance_ratio_.sum()), 3))

variance kept by the 2 PCA axes: 0.391


In [4]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))
for a, Z, name in zip(ax, (P, T), ('PCA', 't-SNE')):
    for i, w in enumerate(words):
        c = '#0b6b5f' if i < 10 else '#c2410c'
        a.scatter(*Z[i], c=c, s=40)
        a.annotate(w, Z[i], xytext=(4, 4), textcoords='offset points', fontsize=9)
    a.set_title(name); a.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUT / 'lab02_embeddings.png', dpi=140); plt.show()

In [5]:
def mean_cos(A, B, same):
    vals = [float(wv.similarity(a, b)) for i, a in enumerate(A) for j, b in enumerate(B) if not (same and i == j)]
    return sum(vals) / len(vals)
within_m, within_s, between = mean_cos(medical, medical, True), mean_cos(sports, sports, True), mean_cos(medical, sports, False)
print(f'mean cosine inside medical {within_m:.3f}, inside sports {within_s:.3f}, between the two {between:.3f}')
print('closest medical pair:', max(((a, b) for i, a in enumerate(medical) for b in medical[i+1:]), key=lambda p: wv.similarity(*p)))
print('words nearest to the sports centroid among the medical list:', sorted(medical, key=lambda w: -float(wv.cosine_similarities(wv[w], [np.mean([wv[s] for s in sports], axis=0)])[0]))[:3])

mean cosine inside medical 0.420, inside sports 0.512, between the two 0.142
closest medical pair: ('doctor', 'nurse')
words nearest to the sports centroid among the medical list: ['doctor', 'surgery', 'hospital']


## Observations

Words from one domain sit closer to each other than to another domain, in both plots. The numbers above back the picture. t-SNE separates the groups more sharply; PCA keeps relative distances. Changing the seed or perplexity moves t-SNE points, so cluster shapes should be read, not distances between clusters.